**Завдання**

На цьому тижні ми вивчили як працюють рекомендаційні системи. Пропонуємо вам познайомитись з бібліотекою surprise, котра якраз є по суті додатком до знайомої нам бібліотеки scikit-learn для тренування моделей рекомендаційних систем.

Візьміть датасет movielens і побудуйте модель матричної факторизації. У даній бібліотеці він має назву SVD. Підберіть найкращі параметри за допомогою крос-валідації, також поекспериментуйте з іншими алгоритмами розрахунків (SVD++, NMF) і оберіть той, який буде оптимальним.

Підказки як саме побудувати дану модель ви знайдете в документації до даної бібліотеки.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from surprise import Dataset, Reader, SVD, accuracy, SVDpp, NMF
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

In [2]:
ratings = pd.read_csv('ratings.csv')

In [3]:
ratings.head(5)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [5]:
ratings.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [6]:
ratings.duplicated().sum()

np.int64(0)

In [7]:
ratings.rating.min(), ratings.rating.max()

(0.5, 5.0)

In [8]:
# Load the dataset into surprise
reader = Reader(rating_scale=(0.5, 5.0))
train_data, test_data = train_test_split(ratings,test_size=0.2,random_state=42)

train_set = Dataset.load_from_df(train_data[['userId','movieId','rating']], reader)
test_set = list(zip(test_data['userId'],
    test_data['movieId'],
    test_data['rating']))

**SVD**
Using SVD model for matrix factorization

In [9]:
param_grid = {"n_epochs": [5, 10,15,20,25], "lr_all": [0.002, 0.005], "reg_all": [0.01, 0.02, 0.05, 0.1]}
grid_svd = GridSearchCV(SVD,param_grid,measures=["rmse", "mae"],cv=3)
grid_svd.fit(train_set)

In [10]:
grid_svd.best_score, grid_svd.best_params

({'rmse': np.float64(0.8759072548292811),
  'mae': np.float64(0.6754366745692127)},
 {'rmse': {'n_epochs': 25, 'lr_all': 0.005, 'reg_all': 0.1},
  'mae': {'n_epochs': 25, 'lr_all': 0.005, 'reg_all': 0.1}})

In [11]:
best_params = grid_svd.best_params['rmse']
model_svd = SVD(**best_params,random_state=42)

trainset = train_set.build_full_trainset()
model_svd.fit(trainset)

In [12]:
predictions_svd = model_svd.test(test_set)
# predictions_svd

In [13]:
rmse_svd = accuracy.rmse(predictions_svd)
mae_svd = accuracy.mae(predictions_svd)
rmse_svd, mae_svd

RMSE: 0.8777
MAE:  0.6719


(np.float64(0.8777139132406059), np.float64(0.6719485228135819))

**SVD++** model for matrix factorization

In [14]:
param_grid = {"n_epochs": [5,10,20], "lr_all": [0.002, 0.005], "reg_all": [0.02, 0.05]}
grid_svdpp = GridSearchCV(SVDpp,param_grid,measures=["rmse", "mae"],cv=3)
grid_svdpp.fit(train_set)

In [15]:
grid_svdpp.best_score, grid_svdpp.best_params

({'rmse': np.float64(0.8752015780751669),
  'mae': np.float64(0.6738914630507874)},
 {'rmse': {'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02},
  'mae': {'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}})

In [16]:
best_params = grid_svdpp.best_params['rmse']
model_svdpp = SVDpp(**best_params, random_state=42)

predictions_svdpp = model_svdpp.fit(trainset).test(test_set)

In [17]:
rmse_svdpp = accuracy.rmse(predictions_svdpp)
mae_svdpp = accuracy.mae(predictions_svdpp)
rmse_svdpp, mae_svdpp

RMSE: 0.8686
MAE:  0.6636


(np.float64(0.8686025514424177), np.float64(0.6636394839075852))

**NMF** model for matrix factorization

In [21]:
param_grid = {"n_epochs": [5,10,20]}
grid_nmf = GridSearchCV(NMF,param_grid,measures=["rmse", "mae"],cv=3)
grid_nmf.fit(train_set)

In [22]:
grid_nmf.best_score, grid_nmf.best_params

({'rmse': np.float64(0.9461809620407092),
  'mae': np.float64(0.7211667628786618)},
 {'rmse': {'n_epochs': 20}, 'mae': {'n_epochs': 20}})

In [23]:
best_params = grid_nmf.best_params['rmse']
model_nmf = NMF(**best_params,random_state=42)

predictions_nmf = model_nmf.fit(trainset).test(test_set)

In [24]:
rmse_nmf = accuracy.rmse(predictions_nmf)
mae_nmf = accuracy.mae(predictions_nmf)
rmse_nmf, mae_nmf

RMSE: 0.9357
MAE:  0.7109


(np.float64(0.9356965054050409), np.float64(0.7109443646983172))

In [25]:
metrics_df = pd.DataFrame(
    {
        'RMSE': [rmse_svd, rmse_svdpp, rmse_nmf],
        'MAE': [mae_svd, mae_svdpp, mae_nmf]
    },
    index=['SVD', 'SVD++', 'NMF']
)
metrics_df

,RMSE,MAE
SVD,0.877714,0.671949
SVD++,0.868603,0.663639
NMF,0.935697,0.710944


Based on the values of rmse and mae metrics, SVD++ model demonstrates the lowest values of both rmse and mae, consequently, model_svdpp is the final choice. 

In [31]:
comparison_df = pd.DataFrame([(pred.uid,pred.iid,pred.r_ui,pred.est) for pred in predictions_svdpp],columns=['User_ID','Movie_ID','Actual_Rating','Predicted_Rating'])
comparison_df

,User_ID,Movie_ID,Actual_Rating,Predicted_Rating
0,432,77866,4.5,3.306690
1,288,474,3.0,3.427612
2,599,4351,3.0,2.586014
3,42,2987,4.0,4.095520
4,75,1610,4.0,3.388419
...,...,...,...,...
20163,380,5048,2.0,3.305656
20164,434,54272,3.5,3.403873
20165,226,5989,4.5,3.870618
20166,607,1320,3.0,3.605544


In [32]:
movies = pd.read_csv('movies.csv')

In [33]:
comparison_df = comparison_df.merge(movies[['movieId', 'title']],
    left_on='Movie_ID',
    right_on='movieId')

In [34]:
comparison_df = comparison_df[
    ['User_ID','title','Actual_Rating','Predicted_Rating']
]
comparison_df

,User_ID,title,Actual_Rating,Predicted_Rating
0,432,Robin Hood (2010),4.5,3.306690
1,288,In the Line of Fire (1993),3.0,3.427612
2,599,Point Break (1991),3.0,2.586014
3,42,Who Framed Roger Rabbit? (1988),4.0,4.095520
4,75,"Hunt for Red October, The (1990)",4.0,3.388419
...,...,...,...,...
20163,380,Snow Dogs (2002),2.0,3.305656
20164,434,"Simpsons Movie, The (2007)",3.5,3.403873
20165,226,Catch Me If You Can (2002),4.5,3.870618
20166,607,Alien³ (a.k.a. Alien 3) (1992),3.0,3.605544


In [35]:
comparison_df['Error'] = (comparison_df['Actual_Rating'] - comparison_df['Predicted_Rating']).abs()
comparison_df

,User_ID,title,Actual_Rating,Predicted_Rating,Error
0,432,Robin Hood (2010),4.5,3.306690,1.193310
1,288,In the Line of Fire (1993),3.0,3.427612,0.427612
2,599,Point Break (1991),3.0,2.586014,0.413986
3,42,Who Framed Roger Rabbit? (1988),4.0,4.095520,0.095520
4,75,"Hunt for Red October, The (1990)",4.0,3.388419,0.611581
...,...,...,...,...,...
20163,380,Snow Dogs (2002),2.0,3.305656,1.305656
20164,434,"Simpsons Movie, The (2007)",3.5,3.403873,0.096127
20165,226,Catch Me If You Can (2002),4.5,3.870618,0.629382
20166,607,Alien³ (a.k.a. Alien 3) (1992),3.0,3.605544,0.605544
